In [1]:
!python3 -m pip install -U python-woc pandas matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['JuliaGaussianProcesses/AbstractGPs.jl', 'pyushkevich/itksnap', 'darrencroton/sage', 'nanograv/enterprise', 'nsidc/mapxmaps', 'kinnala/scikit-fem', 'alibaba/GraphScope', 'TuringLang/Bijectors.jl', 'sandialabs/pecos', 'sicara/tuto-fsl']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,0042246a0dd64b15adbeea4a0c71a81889dab790,juliagaussianprocesses_abstractgps.jl


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  6%|███████▋                                                                                                                       | 198/3294 [03:24<54:04,  1.05s/it]

Got Errors {'1cd311bbbdb6c340a494d4689c1f2b7e4ce82121': 'Key 1cd311bbbdb6c340a494d4689c1f2b7e4ce82121 not found in /da5_fast/All.sha1c/commit_28.tch'}


  6%|███████▋                                                                                                                       | 199/3294 [03:25<53:56,  1.05s/it]

Got Errors {'1d57c3f5123d4672083233b1dd8322242dc257d1': 'Key 1d57c3f5123d4672083233b1dd8322242dc257d1 not found in /da5_fast/All.sha1c/commit_29.tch'}


  7%|████████▍                                                                                                                      | 219/3294 [03:46<53:30,  1.04s/it]

...

Got Errors {'afd119394ef881ee5d003629028bb66e1591269d': 'Key afd119394ef881ee5d003629028bb66e1591269d not found in /da5_fast/All.sha1c/commit_47.tch'}


 99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎| 3276/3294 [57:20<00:18,  1.04s/it]

Got Errors {'fe516c8b16c7c12ec345c78928bfebfefda7ebea': 'Key fe516c8b16c7c12ec345c78928bfebfefda7ebea not found in /da5_fast/All.sha1c/commit_126.tch'}


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3294/3294 [57:39<00:00,  1.05s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0042246a0dd64b15adbeea4a0c71a81889dab790,9e111407324e3488fc1baabd994a910a66b6af52,"[669c98590f1720c0a06eea1f0923e0c67cdaefb0, 1c3...",ST John <st--@users.noreply.github.com>,1649313401,+0300,ST John <st--@users.noreply.github.com>,1649313401,+0300,Merge branch 'st/ad_tests2' of github.com:Juli...,0042246a0dd64b15adbeea4a0c71a81889dab790,juliagaussianprocesses_abstractgps.jl
1,00783f9701edb3717125aa16b9e6922e27556595,a169fbf77e10fc6ca25eef05f6119d8b5570c454,[fa67baea5bebacef20a28eb73508d2d44dbb860c],Sharan Yalburgi <sharanyalburgi@gmail.com>,1595875021,+0530,Sharan Yalburgi <sharanyalburgi@gmail.com>,1595875021,+0530,Style fixes\n,00783f9701edb3717125aa16b9e6922e27556595,juliagaussianprocesses_abstractgps.jl


In [4]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0042246a0dd64b15adbeea4a0c71a81889dab790', 'tree': '9e111407324e3488fc1baabd994a910a66b6af52', 'parent': ['669c98590f1720c0a06eea1f0923e0c67cdaefb0', '1c3366b19b7554a669f5b44be01fb0686ea6d413'], 'author': 'ST John <st--@users.noreply.github.com>', 'author_time': 1649313401, 'author_tz': '+0300', 'committer': 'ST John <st--@users.noreply.github.com>', 'committer_time': 1649313401, 'committer_tz': '+0300', 'message': "Merge branch 'st/ad_tests2' of github.com:JuliaGaussianProcesses/AbstractGPs.jl into st/ad_tests2\n"}


In [5]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [6]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,sicara_tuto-fsl,0042246a0dd64b15adbeea4a0c71a81889dab790,ST John <st--@users.noreply.github.com>,1649313401,Merge branch 'st/ad_tests2' of github.com:Juli...


In [7]:
#mode a means append, so you have all your projects in the same file
yournetid='kbissonn'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github